In [1]:
import pandas as pd

In [2]:
# load datasets
households = pd.read_csv("train_val_households.csv")
persons = pd.read_csv("train_val_persons.csv")
trips = pd.read_csv("train_val_trips.csv")

In [3]:
# merge person -> household
persons_full = persons.merge(households, on="hhid", how="left")

In [4]:
# merge trips -> persons
data = trips.merge(persons_full, on=["hhid", "persid"], how="left")

In [5]:
print(data.shape)
print(data.head())

(13641, 62)
   tripid  hhid  persid  tripno  starthour  startime  arrhour  arrtime  \
0       0     0       0       1          9       545        9      548   
1       1     0       0       2          9       589        9      591   
2       2     0       0       3         11       673       11      675   
3       3     0       0       4         11       683       11      684   
4       4     0       0       5         12       745       12      747   

        origplace1       destplace1  ...  totalvehs  travmonth  travdow_y  \
0      Other place  Outdoor feature  ...          1   February     Sunday   
1  Outdoor feature    Accommodation  ...          1   February     Sunday   
2    Accommodation  Outdoor feature  ...          1   February     Sunday   
3  Outdoor feature  Outdoor feature  ...          1   February     Sunday   
4  Outdoor feature    Accommodation  ...          1   February     Sunday   

       homelga  youngestgroup_5 aveagegroup_5  oldestgroup_5  \
0  Darebin (C)  

In [6]:
X = data.drop(columns=["mode", "tripid"])
y = data["mode"]

In [10]:
# baseline model
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer  # Import the imputer

# Assuming X is your feature dataframe
# First, identify categorical columns (those containing strings like 'Accommodation')
categorical_columns = X.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_columns = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Create preprocessing steps for categorical and numerical data
# Add imputation steps for both categorical and numerical features
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),  # Impute categorical with most frequent value
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_columns),
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='mean'))  # Impute numerical with mean
        ]), numerical_columns)
    ])

# Create a pipeline with preprocessing and the model
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=200))
])

# Split the data
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Fit the pipeline
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['origplace1', 'destplace1',
                                                   'trippurp', 'travdow_x',
                                                   'agegroup', 'sex',
                                                   'relationship', 'carlicence',
                                                   'mbikelicence',
                                                   'otherlicence', 'nolicence',
                                                   'fulltimework',
                                                   'partti...
                                                   'anzsic1', 'startplace',
                                                   'numstops', 'persinc',
                                                   'anywfh', 'wfhmon', 'wfhtue',
                                                   'wfhwed', 'wfhthu', 'wfhfri', ...]),
                                                 ('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer())]),
                                                  ['hhid', 'persid', 'tripno',
                                                   'starthour', 'startime',
                                                   'arrhour', 'arrtime',
                                                   'triptime', 'travtime',
                                                   'cumdist', 'duration',
                                                   'persno', 'hhsize',
                                                   'totalvehs'])])),
                ('classifier', RandomForestClassifier(n_estimators=200))])

In [11]:
# evaluate model on validation data
from sklearn.metrics import accuracy_score, classification_report

y_pred = pipeline.predict(X_val)

print("Accuracy:", accuracy_score(y_val, y_pred))
print(classification_report(y_val, y_pred))

Accuracy: 0.8911689263466471
                 precision    recall  f1-score   support

          CYCLE       1.00      0.42      0.59        52
          DRIVE       0.87      0.98      0.93      1447
          OTHER       0.86      0.32      0.46        19
      PASSENGER       0.91      0.81      0.86       643
PUBLICTRANSPORT       0.90      0.61      0.73       116
           WALK       0.93      0.85      0.89       452

       accuracy                           0.89      2729
      macro avg       0.91      0.67      0.74      2729
   weighted avg       0.89      0.89      0.89      2729



In [12]:
# compute logloss
from sklearn.metrics import log_loss

y_prob = pipeline.predict_proba(X_val)

print("Log Loss:", log_loss(y_val, y_prob))

Log Loss: 0.4067701690625323


In [13]:
# class order
pipeline.classes_

array(['CYCLE', 'DRIVE', 'OTHER', 'PASSENGER', 'PUBLICTRANSPORT', 'WALK'],
      dtype=object)

In [14]:
# train on full training dataset
pipeline.fit(X, y)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['origplace1', 'destplace1',
                                                   'trippurp', 'travdow_x',
                                                   'agegroup', 'sex',
                                                   'relationship', 'carlicence',
                                                   'mbikelicence',
                                                   'otherlicence', 'nolicence',
                                                   'fulltimework',
                                                   'partti...
                                                   'anzsic1', 'startplace',
                                                   'numstops', 'persinc',
                                                   'anywfh', 'wfhmon', 'wfhtue',
                                                   'wfhwed', 'wfhthu', 'wfhfri', ...]),
                                                 ('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer())]),
                                                  ['hhid', 'persid', 'tripno',
                                                   'starthour', 'startime',
                                                   'arrhour', 'arrtime',
                                                   'triptime', 'travtime',
                                                   'cumdist', 'duration',
                                                   'persno', 'hhsize',
                                                   'totalvehs'])])),
                ('classifier', RandomForestClassifier(n_estimators=200))])